# Лекция: Дискриминантный анализ в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 14** (адаптация с языка R на Python)

## Краткая теория

**Дискриминантный анализ (LDA)** — метод классификации: по известным признакам отнести объект к одному из заранее заданных классов.

Отличия от кластеризации: классы **известны** (обучение с учителем).

Классический пример Фишера (1936): ирисы setosa / versicolor / virginica.

В R: `lda()` (MASS), `predict()`, `manova`  
В Python: `sklearn.discriminant_analysis.LinearDiscriminantAnalysis`, PCA.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from statsmodels.multivariate.manova import MANOVA

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Датасет Iris

В R: `library(MASS); View(iris)`


In [ ]:
iris_raw = load_iris()
iris = pd.DataFrame(iris_raw.data, columns=iris_raw.feature_names)
iris["Species"] = pd.Categorical.from_codes(iris_raw.target, iris_raw.target_names)
iris.columns = ["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width", "Species"]
print(iris.head())
print(iris["Species"].value_counts())
print(iris.describe().round(2))


In [ ]:
sns.pairplot(iris, hue="Species", diag_kind="hist", height=2)
plt.suptitle("Iris: попарные диаграммы", y=1.02)
plt.show()


---
## 2. PCA (разведка данных)

В R: `prcomp`, `biplot`, `loadings`


In [ ]:
X = iris.iloc[:, :4]
y = iris["Species"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
scores = pca.fit_transform(X_scaled)

print("Доля объяснённой дисперсии:", np.round(pca.explained_variance_ratio_, 3))
print("Накопленная:", np.round(np.cumsum(pca.explained_variance_ratio_), 3))
print("\nLoadings:")
print(pd.DataFrame(pca.components_.T, index=X.columns,
                   columns=[f"PC{i+1}" for i in range(4)]).round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for sp in y.unique():
    mask = y == sp
    ax.scatter(scores[mask, 0], scores[mask, 1], label=sp, alpha=0.7, s=50)

load = pca.components_.T * np.sqrt(pca.explained_variance_)
for i, name in enumerate(X.columns):
    ax.arrow(0, 0, load[i, 0]*2.5, load[i, 1]*2.5, color="k", alpha=0.6, head_width=0.08)
    ax.text(load[i, 0]*2.7, load[i, 1]*2.7, name, fontsize=9)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.legend()
ax.set_title("PCA biplot (Iris)")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()


---
## 3. LDA: обучение и прогноз

В R: `lda()`, `predict()`, `table()`


In [ ]:
idx_train = np.arange(0, len(iris), 5)
idx_test = np.setdiff1d(np.arange(len(iris)), idx_train)

X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
X_test, y_test = X.iloc[idx_test], y.iloc[idx_test]
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

lda = LDA()
lda.fit(X_train, y_train)

y_pred = lda.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred).round(3))
print("\nConfusion matrix:")
print(pd.crosstab(y_pred, y_test, rownames=["Predicted"], colnames=["Actual"]))
print("\n", classification_report(y_test, y_pred))


In [ ]:
print("Классы:", lda.classes_)
print("Приоры:", np.round(lda.priors_, 3))
print("\nСредние по классам:")
print(pd.DataFrame(lda.means_, index=lda.classes_, columns=X.columns).round(2))
print("\nКоэффициенты LD:")
print(pd.DataFrame(lda.scalings_, index=X.columns,
                   columns=[f"LD{i+1}" for i in range(lda.scalings_.shape[1])]).round(3))


### График LD1–LD2


In [ ]:
X_lda_all = lda.transform(X)
fig, ax = plt.subplots(figsize=(8, 6))
for sp in y.unique():
    mask = y == sp
    ax.scatter(X_lda_all[mask, 0], X_lda_all[mask, 1], label=sp, s=50, alpha=0.8)
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.legend()
ax.set_title("LDA: проекция Iris на LD1–LD2")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()


### Проверка разделимости: MANOVA (Wilks)

H0: средние векторы классов совпадают.


In [ ]:
df_m = X_test.copy()
df_m["class"] = y_pred.astype(str)
maov = MANOVA.from_formula(
    "Q('Sepal.Length') + Q('Sepal.Width') + Q('Petal.Length') + Q('Petal.Width') ~ class",
    data=df_m
)
print(maov.mv_test())


---
## 4. Задание: INZs.csv

```python
df = pd.read_csv("INZs.csv")
X = df.select_dtypes(include=[np.number])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
lda = LDA().fit(X_train, y_train)
print(classification_report(y_test, lda.predict(X_test)))
```


In [ ]:
np.random.seed(0)
n_per = 40
demo = pd.DataFrame({
    "x1": np.concatenate([np.random.normal(0, 1, n_per), np.random.normal(2, 1, n_per), np.random.normal(0, 1, n_per)]),
    "x2": np.concatenate([np.random.normal(0, 1, n_per), np.random.normal(0, 1, n_per), np.random.normal(2, 1, n_per)]),
    "x3": np.concatenate([np.random.normal(1, 1, n_per), np.random.normal(-1, 1, n_per), np.random.normal(0, 1, n_per)]),
    "Class": np.repeat(["A", "B", "C"], n_per),
})

Xd, yd = demo[["x1", "x2", "x3"]], demo["Class"]
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=42, stratify=yd)
lda_d = LDA().fit(Xtr, ytr)
pred = lda_d.predict(Xte)
print("Accuracy:", accuracy_score(yte, pred).round(3))
print(pd.crosstab(pred, yte, rownames=["Pred"], colnames=["True"]))

coords = lda_d.transform(Xd)
plt.figure(figsize=(7, 5))
for c in yd.unique():
    m = yd == c
    plt.scatter(coords[m, 0], coords[m, 1], label=c, s=50)
plt.xlabel("LD1"); plt.ylabel("LD2")
plt.title("Demo LDA")
plt.legend()
plt.tight_layout()
plt.show()


### Как описать результаты

1. Accuracy на тестовой выборке.
2. Confusion matrix: какие классы путаются.
3. Wilks / MANOVA: p < 0.05 → классы различаются.
4. Интерпретация LD (scalings).

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `lda(x, grouping)` | `LDA().fit(X, y)` |
| `predict(lda)$class` | `lda.predict(X)` |
| `predict(lda)$posterior` | `lda.predict_proba(X)` |
| `predict(lda)$x` | `lda.transform(X)` |
| `table(pred, true)` | `pd.crosstab` / `confusion_matrix` |
| `prcomp` / biplot | `PCA` + scatter + arrows |
| `manova` + Wilks | `MANOVA.from_formula` |

---
## Рекомендации

1. Стандартизация признаков при разном масштабе.
2. LDA предполагает нормальность и равные ковариации.
3. Файл **INZs.csv** подставьте локально.
4. PCA ищет дисперсию, LDA — **разделимость классов**.

**Удачи с выполнением Задания 14!**
